In [2]:
import numpy as np
import torch
import torch.utils.data as data
import torch.nn as nn
import sys
import warnings
from itertools import product

### NoteBook environment specific (has to be configured on user side)

In [3]:
# import github repo into notebook file directory, might require fixing path
!rm -r /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

# path to the github repo
root_dir = "/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection"
sys.path.append(root_dir)

import utils.config as config
# arg1: path to the cats dataset, arg2: path to the checkpoint folder
config.init("/kaggle/input/datasets/kirillvishnyakov/cats-dataset/data.csv", root_dir + "/checkpoint_dir")

Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 432, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 432 (delta 37), reused 56 (delta 19), pack-reused 356 (from 1)
Receiving objects: 100% (432/432), 119.01 MiB | 38.19 MiB/s, done.
Resolving deltas: 100% (223/223), done.
70674e6 (HEAD -> main, origin/main, origin/HEAD) typo


### Setup the environment to have access to required repo functions

In [4]:
warnings.filterwarnings("ignore")
from models.lstm_ae import lstm_ae
import dataset
import train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(device)

2.9.0+cu126
cuda


### Random states for reproducibility

In [5]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## tuning helper

In [ ]:
def tune_over_grid(hyperparam_grid, train_dataset, val_dataset, epochs = 10):
    results = []
    
    for lookback_window, embedding_dim_ratio, learning_rate in product(
        hyperparam_grid["lookback_window"],
        hyperparam_grid["embedding_dim_ratio"],
        hyperparam_grid["learning_rate"]
    ):
    
        model = lstm_ae(17, lookback_window, embedding_dim_ratio).to(device)
    
        name = f"lb_{lookback_window}_edr_{embedding_dim_ratio}_lr_{learning_rate}"
    
        weights, val_loss, train_loss = train.fit(
            device, model, name, train_dataset, val_dataset, learning_rate, 512, epochs
        )
    
        results.append({
            "val_loss": val_loss,
            "name": name,
            "weights": weights
        })
    results.sort(key=lambda x: x['val_loss'])
    return results

In [7]:
train_dataset = dataset.AE_Dataset(device, 256, start=0, end=800000)
val_dataset = dataset.AE_Dataset(device, 256, scaler=train_dataset.scaler, start=800000, end=1000000, train=False)

## Tune over latent space ratios

In [8]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [2.0, 2.5, 3.0], 
  "learning_rate": [0.001]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset)
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 15620 total steps
|lb_256_edr_2.0_lr_0.001| train = 0.0299 | test= 0.0228 | LR: 9.93e-04
|lb_256_edr_2.0_lr_0.001| train = 0.0024 | test= 0.0095 | LR: 9.40e-04
|lb_256_edr_2.0_lr_0.001| train = 0.0017 | test= 0.0055 | LR: 8.40e-04
|lb_256_edr_2.0_lr_0.001| train = 0.0013 | test= 0.0031 | LR: 7.04e-04
|lb_256_edr_2.0_lr_0.001| train = 0.0012 | test= 0.0032 | LR: 5.46e-04
|lb_256_edr_2.0_lr_0.001| train = 0.0012 | test= 0.0028 | LR: 3.83e-04
|lb_256_edr_2.0_lr_0.001| train = 0.0011 | test= 0.0027 | LR: 2.34e-04
|lb_256_edr_2.0_lr_0.001| train = 0.0011 | test= 0.0023 | LR: 1.14e-04
|lb_256_edr_2.0_lr_0.001| train = 0.0010 | test= 0.0021 | LR: 3.68e-05
|lb_256_edr_2.0_lr_0.001| train = 0.0010 | test= 0.0020 | LR: 1.00e-05
LR Scheduler: 781 warmup steps, 15620 total steps
|lb_256_edr_2.5_lr_0.001| train = 0.0300 | test= 0.0607 | LR: 9.93e-04
|lb_256_edr_2.5_lr_0.001| train = 0.0022 | test= 0.0418 | LR: 9.40e-04
|lb_256_edr_2.5_lr_0.001| train = 0.0017 | test=

## Test lr - higher ones since training is stable and loss is still decreasing. <br>edr 3.0 and edr 2.0 both converge to ~0.0020. Choosing to stick to 3.0

In [9]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [3.0], 
  "learning_rate": [0.0015, 0.002, 0.003]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset, epochs = 15)
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 23430 total steps
|lb_256_edr_3.0_lr_0.0015| train = 0.0233 | test= 0.0375 | LR: 1.50e-03
|lb_256_edr_3.0_lr_0.0015| train = 0.0020 | test= 0.0048 | LR: 1.46e-03
|lb_256_edr_3.0_lr_0.0015| train = 0.0015 | test= 0.0027 | LR: 1.39e-03
|lb_256_edr_3.0_lr_0.0015| train = 0.0014 | test= 0.0022 | LR: 1.30e-03
|lb_256_edr_3.0_lr_0.0015| train = 0.0012 | test= 0.0018 | LR: 1.17e-03
|lb_256_edr_3.0_lr_0.0015| train = 0.0012 | test= 0.0020 | LR: 1.03e-03
|lb_256_edr_3.0_lr_0.0015| train = 0.0012 | test= 0.0015 | LR: 8.78e-04
|lb_256_edr_3.0_lr_0.0015| train = 0.0011 | test= 0.0016 | LR: 7.17e-04
|lb_256_edr_3.0_lr_0.0015| train = 0.0011 | test= 0.0012 | LR: 5.59e-04
|lb_256_edr_3.0_lr_0.0015| train = 0.0010 | test= 0.0013 | LR: 4.10e-04
|lb_256_edr_3.0_lr_0.0015| train = 0.0010 | test= 0.0012 | LR: 2.77e-04
|lb_256_edr_3.0_lr_0.0015| train = 0.0010 | test= 0.0011 | LR: 1.66e-04
|lb_256_edr_3.0_lr_0.0015| train = 0.0010 | test= 0.0010 | LR: 8.36e-05
|lb_256_edr_3.

## Train final model with more epochs - slightly lower lr to account for the cosine decay landscape keeping a higher lr for longer because this time the model is training for 30 epochs

In [8]:
hyperparam_grid = \
{ "lookback_window": [256], 
  "embedding_dim_ratio": [3.0], 
  "learning_rate": [0.00225]
}

results = tune_over_grid(hyperparam_grid, train_dataset, val_dataset, epochs = 30)
results_best = results[0]
print(f"best model: {results_best['name']}, {results_best['val_loss']}")

LR Scheduler: 781 warmup steps, 46860 total steps
|lb_256_edr_3.0_lr_0.00225| train = 0.0196 | test= 0.0068 | LR: 2.25e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0017 | test= 0.0030 | LR: 2.24e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0013 | test= 0.0023 | LR: 2.21e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0014 | test= 0.0022 | LR: 2.17e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0013 | test= 0.0018 | LR: 2.12e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0013 | test= 0.0017 | LR: 2.06e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0013 | test= 0.0015 | LR: 1.99e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0013 | test= 0.0017 | LR: 1.91e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0012 | test= 0.0016 | LR: 1.82e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0012 | test= 0.0015 | LR: 1.73e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0012 | test= 0.0014 | LR: 1.62e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0012 | test= 0.0015 | LR: 1.51e-03
|lb_256_edr_3.0_lr_0.00225| train = 0.0012 | test= 0.0015 | LR: 1.40e-03
|

In [ ]:
torch.save(results_best["weights"], 'lstm_AutoEncoder_weights.pth')